# 쌤쌤 세션 3 — FootMR Colab 노트북

GVHMR을 따로 설치할 필요 없음 — FootMR 저장소 자체가 GVHMR 파이프라인 전체(3D 월드 복원)를 포함하고, 그 위에 발 보정까지 한 번에 해준다(`tools/demo.py` 하나로 끝). 근거: `samsam_dev_spec.md`, `samsam_plan.md` 세션 3.

## 시작 전 체크리스트
- [ ] 런타임 유형을 GPU(T4)로 변경했는가 (상단 메뉴 런타임 > 런타임 유형 변경)
- [ ] SMPL(smpl.is.tue.mpg.de, "for Python users" 패키지)·SMPL-X(smpl-x.is.tue.mpg.de) 가입 + 다운로드 완료했는가
- [ ] 촬영한 입력 영상 파일 준비됐는가 (고정 카메라 — `samsam_shooting_guide.md` 기준)

체크포인트를 매번 다시 받지 않도록 Google Drive에 저장해두는 걸 권장(아래 2번 셀).

In [ ]:
# 1. GPU 확인 — T4가 보여야 정상
!nvidia-smi

In [ ]:
# 2. Google Drive 마운트 (체크포인트를 여기 저장해두면 다음 세션에 재다운로드 안 해도 됨)
from google.colab import drive
drive.mount('/content/drive')

import os
CKPT_CACHE = '/content/drive/MyDrive/samsam_footmr_checkpoints'
os.makedirs(CKPT_CACHE, exist_ok=True)
print(f'체크포인트 캐시 폴더: {CKPT_CACHE}')

In [ ]:
# 3. FootMR clone (GVHMR 전체를 포함하므로 별도 GVHMR clone 불필요)
!git clone --recursive --depth 1 --shallow-submodules https://github.com/twehrbein/FootMR.git
%cd FootMR

In [ ]:
# 4. python3.10에 pip 직접 부트스트랩 + 의존성 설치
# 1차 시도 실패 원인: apt 패키지 인덱스가 오래돼 python3.10-venv/dev가 404났음(2026-07-23 확인).
# apt-get update로 인덱스 갱신 + venv 대신 get-pip.py로 부트스트랩(더 견고, apt venv 패키지 불필요).
!rm -rf /content/py310env  # 이전 시도의 깨진 venv 정리
!apt-get update -qq
!apt-get install -y python3.10 python3.10-distutils python3.10-tk > /dev/null
# python3.10-tk: FootMR의 body_model.py 첫 줄이 `from turtle import forward`라는 죽은(사용 안 하는)
# import를 갖고 있는데, turtle 모듈이 tkinter를 요구해서 미리 깔아둔다(2026-07-23 실제로 걸린 문제).
!curl -sS https://bootstrap.pypa.io/get-pip.py -o /tmp/get-pip.py
!python3.10 /tmp/get-pip.py
!python3.10 -m pip install -q --upgrade pip
# chumpy가 pip의 격리된 빌드 환경에 numpy가 없어서 빌드 실패하는 문제(2026-07-23 확인) —
# numpy를 먼저 깔고 --no-build-isolation으로 그 numpy를 보게 해서 우회.
!python3.10 -m pip install numpy==1.23.5
!python3.10 -m pip install --no-build-isolation chumpy
!python3.10 -m pip install -r requirements.txt
!python3.10 -m pip install -e .

## 5. 체크포인트 배치

필요한 최종 구조 (`docs/INSTALL.md` 확인):
```
inputs/checkpoints/
├── body_models/smpl/SMPL_{GENDER}.pkl        # ← 직접 가입해서 받은 파일, 아래 6번에서 업로드
├── body_models/smplx/SMPLX_{GENDER}.npz      # ← 직접 가입해서 받은 파일, 아래 6번에서 업로드
├── dpvo/dpvo.pth                              # ← GVHMR Google Drive 폴더, 아래 7번
├── footmr/footmr_checkpoint.ckpt              # ← FootMR 전용 링크, 아래 7번
├── hmr2/epoch=10-step=25000.ckpt              # ← GVHMR Google Drive 폴더
├── vitpose/vitpose-h-multi-coco.pth           # ← GVHMR Google Drive 폴더
└── yolo/yolov8x.pt                            # ← GVHMR Google Drive 폴더
```
`--use_sapiens` 옵션(더 정확하지만 느림)은 이번엔 생략 — 기본 ViTPose로 충분.

In [ ]:
# 6. SMPL/SMPL-X 본체 파일 업로드 — 미리 Drive 캐시 폴더에 넣어뒀다면 그냥 복사,
# 아니면 로컬에서 직접 업로드(파일 선택 창이 뜬다).
import os, shutil

os.makedirs('inputs/checkpoints/body_models/smpl', exist_ok=True)
os.makedirs('inputs/checkpoints/body_models/smplx', exist_ok=True)

cached_smpl = os.path.join(CKPT_CACHE, 'body_models')
if os.path.isdir(cached_smpl):
    shutil.copytree(cached_smpl, 'inputs/checkpoints/body_models', dirs_exist_ok=True)
    print('Drive 캐시에서 SMPL/SMPL-X 복사 완료')
else:
    print('Drive 캐시에 없음 — 아래에서 직접 업로드하세요 (SMPL_*.pkl, SMPLX_*.npz)')
    from google.colab import files
    uploaded = files.upload()
    for name, _ in uploaded.items():
        dst = 'inputs/checkpoints/body_models/smplx' if 'SMPLX' in name.upper() else 'inputs/checkpoints/body_models/smpl'
        shutil.move(name, os.path.join(dst, name))
    # 다음 세션을 위해 Drive에 캐싱
    shutil.copytree('inputs/checkpoints/body_models', cached_smpl, dirs_exist_ok=True)
    print('업로드 완료 + Drive 캐시 저장')

## 7. 나머지 사전학습 체크포인트

- GVHMR 쪽(dpvo/hmr2/vitpose-h-multi-coco/yolo): [Google Drive 폴더](https://drive.google.com/drive/folders/1eebJ13FUEXrKBawHpJroW0sNSxLjh9xD) — 아래 gdown 셀로 시도, 안 되면 직접 다운로드 후 6번처럼 업로드.
- FootMR 전용(`footmr_checkpoint.ckpt` **그리고 `vitpose-h-wholebody.pth`도 여기 있음** — GVHMR 폴더엔 multi-coco만 있고 wholebody는 없어서 2026-07-23에 한 번 막혔음): [FootMR 데이터 링크](https://cloud.tnt.uni-hannover.de/index.php/s/tpLX3F6Mz4FqHaD) — Nextcloud 공유 링크라 gdown이 안 먹을 가능성 높음, 수동 다운로드 후 업로드 권장. 안 되면 Google Drive에 먼저 올려두고 `os.symlink()`로 연결해도 됨(대용량 파일 재업로드 방지).

In [ ]:
# 7a. GVHMR 체크포인트 폴더 통째로 시도 (구글이 대용량 폴더 다운로드를 종종 막는다 —
# 실패하면 링크를 브라우저로 열어 개별 파일을 받은 뒤 6번 셀처럼 업로드).
!pip install -q gdown
!gdown --folder https://drive.google.com/drive/folders/1eebJ13FUEXrKBawHpJroW0sNSxLjh9xD -O /tmp/gvhmr_ckpts

import shutil, glob, os
for sub in ['dpvo', 'hmr2', 'vitpose', 'yolo']:
    os.makedirs(f'inputs/checkpoints/{sub}', exist_ok=True)
found = glob.glob('/tmp/gvhmr_ckpts/**/*', recursive=True)
print(f'{len(found)}개 파일 다운로드됨 — inputs/checkpoints/ 하위 폴더에 맞게 직접 이동시키세요:')
for f in found:
    print(' ', f)

In [ ]:
# 7b. FootMR 전용 체크포인트 — footmr_checkpoint.ckpt "그리고" vitpose-h-wholebody.pth
# (GVHMR 쪽 Drive 폴더엔 vitpose-h-multi-coco.pth만 있고 wholebody는 없음, 2026-07-23 확인).
# Nextcloud 링크는 보통 수동 다운로드가 안전. 로컬에 받아뒀다면 여기서 업로드:
import os, shutil
os.makedirs('inputs/checkpoints/footmr', exist_ok=True)
os.makedirs('inputs/checkpoints/vitpose', exist_ok=True)
from google.colab import files
uploaded = files.upload()  # footmr_checkpoint.ckpt와 vitpose-h-wholebody.pth 둘 다 선택
for name, _ in uploaded.items():
    dst = 'inputs/checkpoints/vitpose' if 'vitpose' in name else 'inputs/checkpoints/footmr'
    shutil.move(name, os.path.join(dst, name))

In [ ]:
# 8. 입력 영상 업로드
from google.colab import files
uploaded = files.upload()
raw_name = list(uploaded.keys())[0]

# 원본 파일명에 (), @, 공백 등이 있으면 셸 파싱도, Hydra의 config override 파싱도 깨진다
# (2026-07-23 둘 다 실제로 걸림 — `@_..._(1920p30).mp4` 같은 유튜브 스크레이핑 파일명 특히 위험).
# 그래서 업로드 직후 바로 안전한 이름으로 통일해서 저장한다.
import os
video_filename = "input_video.mp4"
os.rename(raw_name, video_filename)
print(f'업로드된 영상: {raw_name} → {video_filename}로 이름 변경')

In [ ]:
# 9. 실행 — 고정 카메라 촬영이므로 -s(정적 카메라, SLAM 생략)로 속도 확보
# 4번 셀에서 의존성을 설치한 python3.10으로 실행해야 한다(기본 !python은 3.12라 안 됨).
# 파일명은 반드시 따옴표로 감싸야 함(8번에서 안전한 이름으로 바꿔놨으니 지금은 필수는 아니지만 습관적으로).
#
# --no_postproc: 기본값(끔)은 미세한 타이밍·관절 디테일을 후처리로 눌러버릴 수 있음(2026-07-23 확인,
# demo.py 자체 문서화된 동작) — 쌤쌤 코어가 "그 사람 특유의 디테일"을 보존해야 하는 목적이라
# 세션 6 실데이터 평가 때는 --no_postproc도 같이 비교해볼 것.
!python3.10 tools/demo.py --video "{video_filename}" -s

## 10. 결과 확인 및 다운로드

- `outputs/demo/{영상이름}/hmr4d_results.pt` — 우리가 필요한 SMPL-X 파라미터(`smpl_params_global`이 world-grounded 버전, `retarget_smpl_to_cmu.py`의 입력으로 쓸 것). PyTorch `torch.load()`로 열리는 dict.
- 같은 폴더에 `*_incam.mp4`(카메라 시점) / `*_global.mp4`(월드 시점) 렌더링 영상도 나옴 — 육안 검수용(`samsam_plan.md` 3-3).
- footskate 보정 전/후 비교는 FootMR 논문 방식과 별개로, 우리가 직접 발목 궤적을 뽑아 `evaluate_icc.py`/`analyze.py` 쪽에서 봐야 함(아직 스크립트 없음, 다음 과제).

In [ ]:
# 결과를 Drive에 백업 (Colab 세션 끊기면 로컬 파일 날아감)
import shutil, glob
out_dirs = glob.glob('outputs/demo/*')
for d in out_dirs:
    dst = f"/content/drive/MyDrive/samsam_footmr_outputs/{d.split('/')[-1]}"
    shutil.copytree(d, dst, dirs_exist_ok=True)
    print(f'백업 완료: {dst}')